In [3]:
import pandas as pd

dim_school = pd.read_csv("../../../data/data_correction/dim_school.csv")
fact_fees = pd.read_csv("fact_school_fees.csv")

In [4]:
dim_school["school_natural_key"] = (
    dim_school["index_number"].astype(str) + "|" +
    dim_school["medium"].astype(str)
)

In [5]:
duplicates = dim_school[
    dim_school.duplicated(subset=["school_natural_key"], keep=False)
]

print(duplicates.sort_values("school_natural_key"))

                                 school_key  \
3994   4035fc5f-342d-466a-b5bc-e5f20758c135   
2864   2dd370e8-0cfb-449e-95d9-a23585177708   
16015  ff8323aa-5fb3-4d91-88f9-47eb3ac0c346   
15468  f6302fcd-4e40-4be8-b431-6e00884c1594   
11728  ba5cfde5-cc05-4006-8de6-65b8f9161ee3   
...                                     ...   
13971  de48372e-d950-4d72-b82b-5f0e59add359   
3549   391f3a4f-0f5b-4393-bc49-aac269479f7a   
900    0de782bd-6d75-4777-8756-00360ded42d2   
13543  d76a5aa6-e722-40b6-8d78-e670e620c2ed   
6306   65437635-8c86-4dcc-9c33-3f0a49e4c75b   

                                  school_id  \
3994   48326702-cc7b-4a1c-a6cd-2a39223ba31b   
2864   a09dfaab-948f-4995-ae0b-9e2a2aaccd85   
16015  ef8e889e-d1a7-4098-a821-0faca1d304da   
15468  3635ca3d-6069-4892-a52a-1f1ea0c2e177   
11728  d035ebc6-49bc-40ad-af50-00034e815b7a   
...                                     ...   
13971  46034b7c-3867-4f70-b2e1-7264d03ec446   
3549   2e408f14-401d-4976-ac79-68976693b482   
900    8c80a

In [6]:
canonical_school = (
    dim_school
    .sort_values("inserted_at")
    .drop_duplicates(subset=["school_natural_key"], keep="first")
)

In [7]:
mapping = dim_school.merge(
    canonical_school[["school_natural_key","school_key"]],
    on="school_natural_key",
    suffixes=("_old","_new")
)

mapping = mapping[["school_key_old","school_key_new"]]

In [8]:
fact_fees = fact_fees.merge(
    mapping,
    left_on="school_key",
    right_on="school_key_old",
    how="left"
)

fact_fees["school_key"] = fact_fees["school_key_new"]

fact_fees = fact_fees.drop(columns=["school_key_old","school_key_new"])

In [9]:
fact_fees = fact_fees.drop_duplicates(
    subset=["school_key","standard_id","academic_year"]
)

In [10]:
canonical_school.to_csv("../../../data/dashboard_local/data/dim_school.csv", index=False)
fact_fees.to_csv("fact_school_fees_clean.csv", index=False)